# Código

In [ ]:
import math
import heapq
from collections import Counter
from decimal import Decimal, getcontext

# Configura a precisão decimal para evitar sobrefluxo (underflow) na Codificação Aritmética
getcontext().prec = 100

# ==========================================
# PARTE 1 - Run-Length Encoding (RLE)
# ==========================================

def rle(texto: str) -> str:
    """
    Comprime uma string utilizando Run-Length Encoding (RLE).
    Retorna a sequência compactada de caracteres.
    """
    if not texto:
        return ""

    resultado = []
    cont = 1

    # 1. Varre o texto contando caracteres repetidos consecutivos
    for i in range(1, len(texto)):
        if texto[i] == texto[i - 1]:
            cont += 1
        else:
            # Regista a quantidade e o caractere correspondente
            resultado.append(f"{cont}{texto[i - 1]}")
            cont = 1

    # 2. Adiciona o último grupo de caracteres
    resultado.append(f"{cont}{texto[-1]}")

    return "".join(resultado)

def rle_decode(texto_comprimido: str) -> str:
    """
    Descomprime uma string RLE para reconstruir o texto original.
    """
    if not texto_comprimido:
        return ""

    resultado = []
    multiplicador = ""

    # 1. Varre a string interpretando os dígitos como contadores
    for char in texto_comprimido:
        if char.isdigit():
            multiplicador += char
        else:
            # 2. Expande o caractere pelo número de repetições acumulado
            resultado.append(char * int(multiplicador))
            multiplicador = ""

    return "".join(resultado)


# ==========================================
# PARTE 2 - Shannon-Fano Coding
# ==========================================

def shannon_fano(texto: str) -> tuple:
    """
    Comprime uma string utilizando a Codificação de Shannon-Fano.
    Retorna a sequência de bits codificada e o dicionário de códigos gerado.
    """
    if not texto:
        return "", {}

    # 1. Calcula as frequências dos caracteres
    freqs = Counter(texto)

    # 2. Ordena os caracteres de forma decrescente por frequência
    ordenado = sorted(freqs.items(), key=lambda x: x[1], reverse=True)
    codes = {char: "" for char, _ in ordenado}

    # 3. Divide recursivamente a lista tentando equilibrar as frequências em duas metades
    def dividir(lista):
        if len(lista) <= 1:
            return

        total = sum(f for _, f in lista)
        soma_esquerda = 0
        min_diff = float('inf')
        ponto_corte = 0

        # Encontra o melhor ponto de divisão (menor diferença de frequências)
        for i, (_, f) in enumerate(lista):
            soma_esquerda += f
            diff = abs(total - 2 * soma_esquerda)
            if diff < min_diff:
                min_diff = diff
                ponto_corte = i

        # Atribui '0' para a primeira metade e '1' para a segunda metade
        for i in range(len(lista)):
            if i <= ponto_corte:
                codes[lista[i][0]] += "0"
            else:
                codes[lista[i][0]] += "1"

        # Executa recursivamente para as partições resultantes
        dividir(lista[:ponto_corte + 1])
        dividir(lista[ponto_corte + 1:])

    dividir(ordenado)

    # 4. Gera a sequência final de bits comprimidos
    encoded_text = "".join(codes[char] for char in texto)

    return encoded_text, codes

def shannon_fano_decode(bits: str, codes: dict) -> str:
    """
    Descomprime uma sequência de bits de Shannon-Fano mapeando reversamente os códigos.
    """
    if not bits:
        return ""

    # 1. Inverte o mapeamento de códigos para facilitar a busca
    reverse_codes = {v: k for k, v in codes.items()}
    texto_decodificado = []
    acumulo = ""

    # 2. Reconstrói o texto símbolo por símbolo
    for bit in bits:
        acumulo += bit
        if acumulo in reverse_codes:
            texto_decodificado.append(reverse_codes[acumulo])
            acumulo = ""

    return "".join(texto_decodificado)


# ==========================================
# PARTE 3 - Huffman Coding (Estático)
# ==========================================

class HuffmanNode:
    """
    Nó para compor a árvore de Huffman.
    """
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    # Define o operador '<' para que a PriorityQueue (heapq) saiba como ordenar os nós
    def __lt__(self, other):
        return self.freq < other.freq

def huffman(texto: str) -> tuple:
    """
    Comprime uma string utilizando Árvore de Huffman e Fila de Prioridade.
    Retorna o texto codificado e o dicionário de códigos.
    """
    if not texto:
        return "", {}

    # 1. Calcula as frequências
    freqs = Counter(texto)

    # Cria uma fila de prioridade (min-heap) com os nós folhas
    heap = [HuffmanNode(char, freq) for char, freq in freqs.items()]
    heapq.heapify(heap) # Transforma a lista numa fila de prioridade

    # Caso especial: apenas um tipo de caractere no texto
    if len(heap) == 1:
        node = heap[0]
        codes = {node.char: "0"}
        return "0" * node.freq, codes

    # 2. Constrói a árvore de Huffman
    while len(heap) > 1:
        # Extrai os dois nós com as menores frequências
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        # Cria um nó interno com a soma das frequências
        merged = HuffmanNode(None, left.freq + right.freq)
        merged.left = left
        merged.right = right

        # Insere o novo nó de volta na fila de prioridade
        heapq.heappush(heap, merged)

    root = heap[0]
    codes = {}

    # 3. Gera os códigos percorrendo a árvore (Textualmente)
    def generate_codes(node, current_code):
        if node is None:
            return

        # Se for uma folha, registra o código
        if node.char is not None:
            codes[node.char] = current_code

        generate_codes(node.left, current_code + "0")
        generate_codes(node.right, current_code + "1")

    generate_codes(root, "")

    # 4. Codifica a mensagem
    encoded_text = "".join(codes[char] for char in texto)

    return encoded_text, codes

def huffman_decode(bits: str, codes: dict) -> str:
    """
    Descomprime uma string codificada em Huffman utilizando o dicionário reverso.
    """
    if not bits:
        return ""

    # 1. Inverte o dicionário de códigos
    reverse_codes = {v: k for k, v in codes.items()}
    texto_decodificado = []
    acumulo = ""

    # 2. Decodifica a string de bits iterativamente
    for bit in bits:
        acumulo += bit
        if acumulo in reverse_codes:
            texto_decodificado.append(reverse_codes[acumulo])
            acumulo = ""

    return "".join(texto_decodificado)


# ==========================================
# PARTE 4 - LZW (Lempel-Ziv-Welch)
# ==========================================

def lzw(texto: str) -> tuple:
    """
    Comprime uma string utilizando o algoritmo LZW.
    Retorna a sequência de códigos inteiros e o tamanho final do dicionário.
    """
    if not texto:
        return [], 0

    # 1. Inicializa o dicionário com a tabela ASCII
    dicionario = {chr(i): i for i in range(256)}
    prox_codigo = 256

    w = ""
    codigos = []

    # 2. Varre o texto e constrói a sequência de códigos
    for c in texto:
        wc = w + c
        if wc in dicionario:
            w = wc
        else:
            # Adiciona o código da string conhecida à saída
            codigos.append(dicionario[w])
            # Adiciona a nova string ao dicionário
            dicionario[wc] = prox_codigo
            prox_codigo += 1
            w = c

    # 3. Adiciona o último código restante
    if w:
        codigos.append(dicionario[w])

    return codigos, len(dicionario)

def lzw_decode(codigos: list) -> str:
    """
    Descomprime uma lista de códigos LZW, reconstruindo o texto original.
    """
    if not codigos:
        return ""

    # 1. Inicializa o dicionário reverso com a tabela ASCII
    dicionario = {i: chr(i) for i in range(256)}
    prox_codigo = 256

    # 2. Processa o primeiro código
    w = chr(codigos[0])
    resultado = [w]

    # 3. Varre os códigos restantes
    for k in codigos[1:]:
        if k in dicionario:
            entrada = dicionario[k]
        elif k == prox_codigo:
            # Caso especial: a sequência w+w[0] ainda não está no dicionário, mas é necessária
            entrada = w + w[0]
        else:
            raise ValueError("Erro de decodificação LZW: Código inválido.")

        resultado.append(entrada)

        # Atualiza o dicionário com o caractere decodificado
        dicionario[prox_codigo] = w + entrada[0]
        prox_codigo += 1
        w = entrada

    return "".join(resultado)


# ==========================================
# PARTE 5 - Huffman Adaptativo
# ==========================================

class AdaptiveHuffmanNode:
    """
    Nó para compor a árvore dinâmica do Huffman Adaptativo.
    """
    def __init__(self, char=None, weight=0):
        self.char = char
        self.weight = weight
        self.left = None
        self.right = None
        self.parent = None

def _get_adaptive_code(node: AdaptiveHuffmanNode) -> str:
    """
    Sobe na árvore a partir de um nó folha até a raiz para extrair o código binário correspondente.
    """
    codigo = ""
    atual = node
    while atual.parent is not None:
        if atual.parent.left == atual:
            codigo = "0" + codigo
        else:
            codigo = "1" + codigo
        atual = atual.parent
    return codigo

def _update_adaptive_tree(root: AdaptiveHuffmanNode, nyt: AdaptiveHuffmanNode, nodes_dict: dict, char: str) -> tuple:
    """
    Atualiza a árvore inserindo o novo caractere no nó NYT ou incrementando pesos existentes.
    Retorna a raiz e o nó NYT atualizados.
    """
    if char in nodes_dict:
        no_atual = nodes_dict[char]
    else:
        # 1. Divide o nó NYT em um novo NYT à esquerda e o novo símbolo à direita
        nyt.left = AdaptiveHuffmanNode(char="NYT", weight=0)
        nyt.right = AdaptiveHuffmanNode(char=char, weight=0)
        nyt.left.parent = nyt
        nyt.right.parent = nyt

        nodes_dict[char] = nyt.right
        nyt.char = None

        no_atual = nyt.right
        nyt = nyt.left

    # 2. Incrementa os pesos subindo até a raiz
    while no_atual is not None:
        no_atual.weight += 1
        no_atual = no_atual.parent

    return root, nyt

def adaptive_huffman(texto: str) -> tuple:
    """
    Comprime uma string utilizando Huffman Adaptativo (árvore dinâmica).
    Retorna a sequência de bits e o tamanho final da árvore em número de nós.
    """
    if not texto:
        return "", 0

    # 1. Inicializa a árvore vazia apenas com o nó NYT
    raiz = AdaptiveHuffmanNode(char="NYT", weight=0)
    nyt = raiz
    nos_simbolos = {}
    bits_saida = ""

    # 2. Processa cada símbolo dinamicamente
    for c in texto:
        if c in nos_simbolos:
            # Emite o código atual do símbolo
            bits_saida += _get_adaptive_code(nos_simbolos[c])
        else:
            # Emite o caminho do NYT + binário ASCII do novo símbolo (8 bits)
            bits_saida += _get_adaptive_code(nyt)
            bits_saida += format(ord(c), '08b')

        # 3. Atualiza a árvore dinamicamente
        raiz, nyt = _update_adaptive_tree(raiz, nyt, nos_simbolos, c)

    # 4. Conta os nós da árvore final
    def contar_nos(node):
        if node is None: return 0
        return 1 + contar_nos(node.left) + contar_nos(node.right)

    return bits_saida, contar_nos(raiz)

def adaptive_huffman_decode(bits: str, tamanho_texto: int) -> str:
    """
    Descomprime uma sequência de bits utilizando Huffman Adaptativo sincronizado.
    """
    if not bits:
        return ""

    # 1. Inicializa a árvore idêntica ao início da compressão
    raiz = AdaptiveHuffmanNode(char="NYT", weight=0)
    nyt = raiz
    nos_simbolos = {}
    texto_decodificado = []

    i = 0
    # 2. Percorre a sequência de bits para reconstruir os dados
    while len(texto_decodificado) < tamanho_texto and i < len(bits):
        no_atual = raiz

        # Desce na árvore lendo os bits
        while no_atual.left is not None and no_atual.right is not None:
            if i >= len(bits): break
            bit = bits[i]
            i += 1
            if bit == '0':
                no_atual = no_atual.left
            else:
                no_atual = no_atual.right

        if no_atual.char == "NYT":
            # Lê os próximos 8 bits para descobrir o novo caractere
            char_bits = bits[i:i+8]
            i += 8
            simbolo = chr(int(char_bits, 2))
        else:
            simbolo = no_atual.char

        texto_decodificado.append(simbolo)

        # 3. Atualiza a árvore em perfeita sincronia com a compressão
        raiz, nyt = _update_adaptive_tree(raiz, nyt, nos_simbolos, simbolo)

    return "".join(texto_decodificado)


# ==========================================
# PARTE 6 - Arithmetic Coding
# ==========================================

def arithmetic_coding(texto: str) -> tuple:
    """
    Comprime uma string utilizando Codificação Aritmética.
    Retorna o valor final do intervalo, o dicionário de acumulados e a estimativa de bits.
    """
    if not texto:
        return 0, {}, 0

    # 1. Calcula as frequências
    freqs = {}
    for c in texto:
        freqs[c] = freqs.get(c, 0) + 1

    # 2. Converte as probabilidades utilizando precisão Decimal
    tamanho = Decimal(len(texto))
    probs = {c: Decimal(f) / tamanho for c, f in freqs.items()}

    # 3. Constrói os intervalos acumulados ordenadamente
    acumulado = {}
    soma = Decimal(0)
    for c in sorted(probs.keys()):
        acumulado[c] = [soma, soma + probs[c]]
        soma += probs[c]

    # 4. Codifica a mensagem num único intervalo numérico progressivo
    low = Decimal(0)
    high = Decimal(1)
    for c in texto:
        range_ = high - low
        high = low + range_ * acumulado[c][1]
        low = low + range_ * acumulado[c][0]

    # Valor médio final que representa unicamente o intervalo gerado
    valor_final = (low + high) / Decimal(2)

    # 5. Estima o tamanho da representação em bits usando Entropia de Shannon
    prob_mensagem = Decimal(1)
    for c in texto:
        prob_mensagem *= probs[c]
    try:
        tamanho_estimado_bits = math.ceil(-math.log2(float(prob_mensagem)))
    except OverflowError:
        tamanho_estimado_bits = len(texto) * 8 # Fallback caso ultrapasse os limites do float

    return valor_final, acumulado, tamanho_estimado_bits

def arithmetic_decode(valor: Decimal, tamanho_original: int, acumulado: dict) -> str:
    """
    Descomprime o valor numérico para reconstruir o texto de entrada original.
    """
    texto_decodificado = []

    # 1. Decodifica iterativamente com base no tamanho do texto original
    for _ in range(tamanho_original):
        for c, (low, high) in acumulado.items():
            if low <= valor < high:
                texto_decodificado.append(c)
                range_ = high - low
                # Remove a faixa atual para expor o próximo símbolo decodificável
                valor = (valor - low) / range_
                break

    return "".join(texto_decodificado)


# ==========================================
# RELATÓRIO DE COMPRESSÃO COMPLETO
# ==========================================

def gerar_relatorio():
    """
    Gera o relatório testando sistematicamente todos os 6 algoritmos de compressão.
    """
    textos = {
        "Texto 1": "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",
        "Texto 2": "BANANABANANABANANABANANA",
        "Texto 3": "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    }

    print("=" * 60)
    print("      RELATÓRIO DE COMPRESSÃO COMPLETO CALCULADO")
    print("=" * 60)

    for nome, txt in textos.items():
        orig_chars = len(txt)
        orig_bits = orig_chars * 8  # ASCII Base

        print(f"\n--- {nome} ---")
        print(f"Conteúdo: {txt}")
        print(f"Tamanho Original: {orig_chars} caracteres ({orig_bits} bits)\n")

        # ---------------- RLE ----------------
        rle_res = rle(txt)
        rle_chars = len(rle_res)
        rle_bits = rle_chars * 8
        rle_ratio = orig_bits / rle_bits if rle_bits else 0
        txt_rle = rle_decode(rle_res)

        print("[RLE]")
        print(f"Resultado          : {rle_res}")
        print(f"Tamanho Comprimido : {rle_chars} caracteres ({rle_bits} bits)")
        print(f"Taxa de Compressão : {rle_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_rle else 'FALHA'}")
        print("-" * 40)

        # ---------------- Shannon-Fano ----------------
        sf_bits_str, sf_codes = shannon_fano(txt)
        sf_bits = len(sf_bits_str)
        sf_ratio = orig_bits / sf_bits if sf_bits else 0
        txt_sf = shannon_fano_decode(sf_bits_str, sf_codes)

        print("[Shannon-Fano]")
        print(f"Códigos            : {sf_codes}")
        print(f"Tamanho Comprimido : {sf_bits} bits")
        print(f"Taxa de Compressão : {sf_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_sf else 'FALHA'}")
        print("-" * 40)

        # ---------------- Huffman Estático ----------------
        huf_bits_str, huf_codes = huffman(txt)
        huf_bits = len(huf_bits_str)
        huf_ratio = orig_bits / huf_bits if huf_bits else 0
        txt_huf = huffman_decode(huf_bits_str, huf_codes)

        print("[Huffman Estático]")
        print(f"Códigos            : {huf_codes}")
        print(f"Tamanho Comprimido : {huf_bits} bits")
        print(f"Taxa de Compressão : {huf_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_huf else 'FALHA'}")
        print("-" * 40)

        # ---------------- LZW ----------------
        lzw_codes, dict_size = lzw(txt)
        lzw_bits = len(lzw_codes) * 12  # 12 bits por código no dicionário dinâmico
        lzw_ratio = orig_bits / lzw_bits if lzw_bits else 0
        txt_lzw = lzw_decode(lzw_codes)

        print("[LZW]")
        print(f"Dicionário Final   : {dict_size} entradas")
        print(f"Sequência Códigos  : {lzw_codes}")
        print(f"Tamanho Estimado   : {lzw_bits} bits (12 bits/código)")
        print(f"Taxa de Compressão : {lzw_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_lzw else 'FALHA'}")
        print("-" * 40)

        # ---------------- Huffman Adaptativo ----------------
        adp_bits_str, tree_size = adaptive_huffman(txt)
        adp_bits = len(adp_bits_str)
        adp_ratio = orig_bits / adp_bits if adp_bits else 0
        txt_adp = adaptive_huffman_decode(adp_bits_str, len(txt))

        print("[Huffman Adaptativo]")
        print(f"Sequência (bits)   : {adp_bits_str[:50]}..." if len(adp_bits_str) > 50 else f"Sequência (bits)   : {adp_bits_str}")
        print(f"Tamanho Comprimido : {adp_bits} bits")
        print(f"Tamanho da Árvore  : {tree_size} nós")
        print(f"Taxa de Compressão : {adp_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_adp else 'FALHA'}")
        print("-" * 40)

        # ---------------- Arithmetic Coding ----------------
        valor_final, acumulado, arith_bits = arithmetic_coding(txt)
        arith_ratio = orig_bits / arith_bits if arith_bits else 0
        txt_arith = arithmetic_decode(valor_final, len(txt), acumulado)

        print("[Arithmetic Coding]")
        print(f"Intervalo (Valor)  : {valor_final:.15f}...")
        print(f"Tamanho Estimado   : {arith_bits} bits")
        print(f"Taxa de Compressão : {arith_ratio:.2f}x")
        print(f"Integridade        : {'OK' if txt == txt_arith else 'FALHA'}")
        print("=" * 60)

if __name__ == "__main__":
    gerar_relatorio()

      RELATÓRIO DE COMPRESSÃO COMPLETO CALCULADO

--- Texto 1 ---
Conteúdo: AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
Tamanho Original: 30 caracteres (240 bits)

[RLE]
Resultado          : 30A
Tamanho Comprimido : 3 caracteres (24 bits)
Taxa de Compressão : 10.00x
Integridade        : OK
----------------------------------------
[Shannon-Fano]
Códigos            : {'A': ''}
Tamanho Comprimido : 0 bits
Taxa de Compressão : 0.00x
Integridade        : FALHA
----------------------------------------
[Huffman Estático]
Códigos            : {'A': '0'}
Tamanho Comprimido : 30 bits
Taxa de Compressão : 8.00x
Integridade        : OK
----------------------------------------
[LZW]
Dicionário Final   : 263 entradas
Sequência Códigos  : [65, 256, 257, 258, 259, 260, 261, 256]
Tamanho Estimado   : 96 bits (12 bits/código)
Taxa de Compressão : 2.50x
Integridade        : OK
----------------------------------------
[Huffman Adaptativo]
Sequência (bits)   : 0100000111111111111111111111111111111
Tamanho Comprimido : 